In [2]:
import math
import requests
import itertools
import folium
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx

from scipy.spatial.distance import cdist
from tqdm import tqdm

sns.set(
    { "figure.figsize": (17, 7) },
    style='ticks',
    palette=sns.color_palette("Set2"),
    color_codes=True,
    font_scale=5
)

plt.rcParams.update({
    "axes.labelsize": 12,  # Axes label font size
})

%config InlineBackend.figure_format = 'retina'
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load data
stops_df = pd.read_csv("data/timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\stops.txt")
stop_times_df = pd.read_csv("data/timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\stop_times.txt")
trips = pd.read_csv("data/timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\trips.txt")

# Convert datatypes
stop_times_df["arrival_time"] = pd.to_timedelta(stop_times_df["arrival_time"])
stop_times_df["departure_time"] = pd.to_timedelta(stop_times_df["departure_time"])

# Merge route id details into stop times
stop_times_df = stop_times_df.merge(
    trips[["route_id", "trip_id"]],
    left_on='trip_id', 
    right_on='trip_id', 
    how='left')

# Isolate Bradford
stops_df = stops_df.loc[(stops_df["stop_lat"] < 54) & (stops_df["stop_lat"] > 53.7)].reset_index(drop=True)
stops_df = stops_df.loc[(stops_df["stop_lon"] < -1.55) & (stops_df["stop_lon"] > -1.95)].reset_index(drop=True)

In [4]:
stop_route_dict = stop_times_df.groupby("stop_id")["route_id"].unique().to_dict()
route_stops_dict = stop_times_df.groupby("route_id")["stop_id"].unique().to_dict()

In [5]:
origin = (53.79, -1.73)
dest = (53.841401, -1.827540)

In [6]:
lon_scale = math.cos(math.radians(origin[0]))

In [7]:
# 1. Create a bidirectional mapping between stop_ids and integer indices
stop_ids = stops_df["stop_id"].values
stop_idx_map = {stop_id: idx for idx, stop_id in enumerate(stop_ids)}

# 2. Extract and pre-scale coordinates
# Scaling lon_scale here means we can use standard Euclidean distance functions directly
lats = stops_df["stop_lat"].values
lons = stops_df["stop_lon"].values * lon_scale
coords = np.column_stack((lats, lons))

# 3. Pre-compute the full N x N distance matrix in C
# This replaces dists_between_stops_dict entirely
stop_dist_matrix = cdist(coords, coords, metric='euclidean')

# Pre-map route stops to their integer indices to avoid repeated dictionary lookups in the loop
route_stops_idx_dict = {
    route: [stop_idx_map[s] for s in stops if s in stop_idx_map] 
    for route, stops in route_stops_dict.items()
}

unique_routes = trips["route_id"].unique()

In [9]:

# Initialize a Directed Graph
G = nx.DiGraph()

# 1. Bulk populate nodes with attributes
# Utilizing the stop_idx_map, lats, and lons vectors established previously
node_attributes = {
    stop_id: {'idx': idx, 'lat': lats[idx], 'lon': lons[idx]}
    for stop_id, idx in stop_idx_map.items()
}
G.add_nodes_from(node_attributes.items())

# 2. Extract sequential edges and assign pre-computed weights
edge_data = []

for route, ordered_stops in route_stops_dict.items():
    # zip(stops[:-1], stops[1:]) creates pairs of consecutive stops in O(N) time
    for u, v in zip(ordered_stops[:-1], ordered_stops[1:]):
        
        try:
            u_idx = stop_idx_map[u]
            v_idx = stop_idx_map[v]
            
            # O(1) retrieval of the spatial distance from the previously generated dense matrix
            edge_distance = stop_dist_matrix[u_idx, v_idx]
            
            # NetworkX will overwrite existing edges by default. 
            # If multiple routes share this segment, you may want to append the route to a list attribute.
            if G.has_edge(u, v):
                # Example: appending route to an existing edge's route list
                # G[u][v]['routes'].append(route) is slower in a loop; better to handle externally
                continue 
                
            edge_data.append((u, v, {'weight': edge_distance, 'route': route}))
        except:
            continue

# 3. Bulk load edges 
G.add_edges_from(edge_data)

In [10]:
# Calculate the geographic center of the network to initialize the map
lats = [data['lat'] for _, data in G.nodes(data=True)]
lons = [data['lon'] for _, data in G.nodes(data=True)]
center_lat = sum(lats) / len(lats)
center_lon = sum(lons) / len(lons)

# Initialize the base map
m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="CartoDB positron")

# 1. Plot the edges (Route segments)
for u, v, edge_data in G.edges(data=True):
    # Extract coordinates for the start and end nodes of the edge
    u_coord = (G.nodes[u]['lat'], G.nodes[u]['lon'])
    v_coord = (G.nodes[v]['lat'], G.nodes[v]['lon'])
    
    folium.PolyLine(
        locations=[u_coord, v_coord],
        color="blue",
        weight=2,
        opacity=0.6,
        tooltip=f"Route: {edge_data.get('route', 'Unknown')}"
    ).add_to(m)

# 2. Plot the nodes (Stops)
for node, node_data in G.nodes(data=True):
    folium.CircleMarker(
        location=(node_data['lat'], node_data['lon']),
        radius=3,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=1.0,
        popup=f"Stop ID: {node}"
    ).add_to(m)

# Save to an HTML file to view in a browser, or render directly if using a Jupyter Notebook
m.save("transit_graph.html")
# m  # Uncomment this line if executing within a Jupyter Notebook to display inline